# Figure 1

Paper panels with figure-specific PDF and CSV exports. Run from top to bottom. Original analysis notebooks are preserved.

## Setup

In [3]:
from pathlib import Path
from tempfile import TemporaryDirectory
from contextlib import contextmanager
from IPython.display import IFrame, display
from matplotlib.transforms import Bbox
import itertools
import os
import shutil
import pandas as pd
import KinematicPlot as kp
from group_config_new import build_groups
from survival_stats_runner import SurvivalStatsRunner

# All persistent exports belong to this paper figure, with one folder per panel.
ROOT = Path.cwd()
if not (ROOT / "KinematicPlot.py").is_file():
    raise RuntimeError("Run this notebook from the repository root.")
FIGURE_NUMBER = 1
NOTEBOOK_OUTPUT_DIR = ROOT / "Figures" / f"Figure{FIGURE_NUMBER}"
SC_DATA_DIR = ROOT / "SC data"
N_PERM = 20000
plotter = kp.PlotCreator()
stats_runner = SurvivalStatsRunner(tau=0.71, random_state=0, platform_offset=0.03, radius=0.07, fps=250)

def output_folder(*parts):
    # Create output folders only when the notebook is executed.
    folder = NOTEBOOK_OUTPUT_DIR.joinpath(*parts)
    folder.mkdir(parents=True, exist_ok=True)
    return folder

def panel_prefix(panel, name):
    return output_folder(f"Figure{panel}") / f"Figure{panel}_{name}"

def show_pdf(path, width=950, height=700):
    display(IFrame(src=Path(path).relative_to(ROOT).as_posix(), width=width, height=height))

@contextmanager
def working_directory(folder):
    # Preserve the working directory for legacy notebook loader calls.
    previous = Path.cwd()
    os.chdir(folder)
    try:
        yield
    finally:
        os.chdir(previous)

In [4]:
# Shared WT tracking QC thresholds for angle and TT trajectory analyses.
# Change WT_QC_ERROR_MAX here to adjust the reprojection-error cutoff for both.
WT_QC_ERROR_MAX = 30
WT_QC_SCORE_MIN = 0.8
WT_QC_MIN_CAMERAS = 2
WT_QC_MAX_INTERP_GAP_S = 0.02
WT_QC_MAX_INVALID_FRACTION = 0.3
WT_QC_MIN_VALID_FRACTION = 1.0 - WT_QC_MAX_INVALID_FRACTION

# Smooth after QC/interpolation with the utilities exponential moving average.
WT_ANGLE_SMOOTH = True
WT_ANGLE_SMOOTH_ALPHA = 0.4

WT_ANGLE_QC_KWARGS = dict(
    apply_tracking_qc=True,
    min_cameras=WT_QC_MIN_CAMERAS,
    max_interp_gap_s=WT_QC_MAX_INTERP_GAP_S,
    min_valid_fraction=WT_QC_MIN_VALID_FRACTION,
    error_max=WT_QC_ERROR_MAX,
    score_min=WT_QC_SCORE_MIN,
    smooth_angle=WT_ANGLE_SMOOTH,
    smooth_alpha=WT_ANGLE_SMOOTH_ALPHA,
)

WT_TT_QC_KWARGS = dict(
    apply_tracking_qc=True,
    min_cameras=WT_QC_MIN_CAMERAS,
    max_interp_gap_s=WT_QC_MAX_INTERP_GAP_S,
    min_valid_fraction=WT_QC_MIN_VALID_FRACTION,
    error_max=WT_QC_ERROR_MAX,
    score_min=WT_QC_SCORE_MIN,
)

WT_ANGLE_QC_KWARGS, WT_TT_QC_KWARGS

({'apply_tracking_qc': True,
  'min_cameras': 2,
  'max_interp_gap_s': 0.02,
  'min_valid_fraction': 0.7,
  'error_max': 30,
  'score_min': 0.8,
  'smooth_angle': True,
  'smooth_alpha': 0.4},
 {'apply_tracking_qc': True,
  'min_cameras': 2,
  'max_interp_gap_s': 0.02,
  'min_valid_fraction': 0.7,
  'error_max': 30,
  'score_min': 0.8})

## WT group

In [6]:
groups = build_groups(group_keys=["WT_T2_TTa"], skip_missing=False, require_kinematics=False)
wt_group_colors = {"WT-T2-TiTa": "#d62728"}

## Figure 1F - Flight postural change

In [8]:
FLIGHT_POSTURE_GROUP_KEY = "WT_T2_TTa"
out = output_folder("Figure1F")
pdf = out / f"Figure1F_{FLIGHT_POSTURE_GROUP_KEY}_R_mFT_pre_MOC_postural_change.pdf"

# Apply the shared WT angle QC rule to only the pre-MOC window that is
# averaged into each fly/trial posture value.
posture_fig, posture_ax, posture_values_df, posture_summary_df, posture_qc_df, posture_skipped_df = plotter.flight_postural_change(
    group_info=groups[FLIGHT_POSTURE_GROUP_KEY],
    angle_def=("R-mCT", "R-mFT", "R-mTT"),
    trial_types=("Landing", "Flying"),
    pre_moc_window_s=1,
    max_trial_num=20,
    colors={groups[FLIGHT_POSTURE_GROUP_KEY].group_name: wt_group_colors[groups[FLIGHT_POSTURE_GROUP_KEY].group_name]},
    show_sem=True,
    file_name=str(pdf.with_suffix("")),
    **WT_ANGLE_QC_KWARGS,
)

display(posture_summary_df)
display(posture_qc_df.head())
display(posture_skipped_df.head())
show_pdf(pdf, height=700)

,Group_Label,Group_Name,Trial#,Joint,Mean_Pre_MOC_Angle_deg,SEM_Pre_MOC_Angle_deg,SD_Pre_MOC_Angle_deg,n_flies,Pre_MOC_Window_s,Apply_Tracking_QC
0,WT-T2-TiTa,WT-T2-TiTa,1,R-mFT,25.865778,1.534062,5.739934,14,1,True
1,WT-T2-TiTa,WT-T2-TiTa,2,R-mFT,40.876386,2.595902,9.712978,14,1,True
2,WT-T2-TiTa,WT-T2-TiTa,3,R-mFT,42.414434,3.304390,12.363897,14,1,True
3,WT-T2-TiTa,WT-T2-TiTa,4,R-mFT,43.302548,2.426127,9.077738,14,1,True
4,WT-T2-TiTa,WT-T2-TiTa,5,R-mFT,46.635636,1.937571,7.249726,14,1,True
5,WT-T2-TiTa,WT-T2-TiTa,6,R-mFT,46.202559,1.682158,6.294060,14,1,True
6,WT-T2-TiTa,WT-T2-TiTa,7,R-mFT,47.257048,2.102415,7.866515,14,1,True
7,WT-T2-TiTa,WT-T2-TiTa,8,R-mFT,51.046817,2.170312,8.120565,14,1,True
8,WT-T2-TiTa,WT-T2-TiTa,9,R-mFT,45.591791,1.533247,5.736884,14,1,True
9,WT-T2-TiTa,WT-T2-TiTa,10,R-mFT,46.425647,2.447279,9.156881,14,1,True


,Joint,Angle_Definition,QC_Passed,QC_Exclusion_Reason,Valid_Frame_Fraction,Invalid_Frame_Fraction,Max_Invalid_Gap_Frames,Interpolated_Frame_Count,Max_Interp_Gap_Frames,Group_Label,Group_Name,Fly#,Trial#,Window_Start_Frame,Window_End_Frame
0,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,105,4,WT-T2-TiTa,WT-T2-TiTa,1,1,283,482
1,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,141,4,WT-T2-TiTa,WT-T2-TiTa,1,2,70,269
2,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,125,4,WT-T2-TiTa,WT-T2-TiTa,1,3,82,281
3,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,105,4,WT-T2-TiTa,WT-T2-TiTa,1,4,107,306
4,R-mFT,R-mCT|R-mFT|R-mTT,True,,1.0,0.0,0,197,4,WT-T2-TiTa,WT-T2-TiTa,1,5,98,297


,Group_Label,Group_Name,Fly#,Trial#,Reason,Joint,Angle_Definition,QC_Passed,QC_Exclusion_Reason,Valid_Frame_Fraction,Invalid_Frame_Fraction,Max_Invalid_Gap_Frames,Interpolated_Frame_Count,Max_Interp_Gap_Frames,Window_Start_Frame,Window_End_Frame
0,WT-T2-TiTa,WT-T2-TiTa,14,13,failed posture tracking QC,R-mFT,R-mCT|R-mFT|R-mTT,False,R-mFT:long_invalid_gap;R-mTT:long_invalid_gap,0.855,0.145,9,117,4,57,256
1,WT-T2-TiTa,WT-T2-TiTa,14,14,failed posture tracking QC,R-mFT,R-mCT|R-mFT|R-mTT,False,R-mFT:long_invalid_gap;R-mTT:long_invalid_gap,0.900,0.100,9,200,4,83,282
